# IVF Pregnancy Prediction Pipeline

## 주요 변경사항

### 모델링
| 항목 | v5 | v6 |
|---|---|---|
| 기저 모델 | LGB + CAT | LGB + CAT + **ExtraTrees** |
| Stacking 메타 모델 | LogisticRegression (OOF 2개 + top10 feature) | **Ridge + LR 앙상블** (OOF 3개 + top feature) |
| Stacking 방식 | 단일 LR | **두 메타 모델 평균** |

### LinearSVM 미채택 이유
LinearSVM은 이 데이터셋에 적합하지 않습니다:
1. **확률값 미지원**: `predict_proba`가 없어 AUC 계산에 Platt Scaling이 필요 (추가 편향)
2. **선형 경계**: IVF 데이터는 나이×배아 수 등 비선형 상호작용이 핵심 → LinearSVM은 이를 포착 불가
3. **스케일 민감**: 이미 트리 기반 모델이 잘 처리하는 카테고리 변수에서 오히려 성능 저하 위험

### ExtraTrees 채택 이유
1. **다양성**: LGB/CAT와 학습 메커니즘이 다름 → 앙상블 다양성 확보
2. **랜덤 분할 임계값**: 특이한 패턴 포착 가능
3. **빠른 학습**: 캐글 세션 시간 절약
4. **확률 출력 지원**: AUC 기반 평가에 바로 사용 가능

### Feature Engineering
기존 13개 유지 + **신규 4개 추가** (논문 근거)

### Data Leakage 수정
- **Stacking 메타 피처의 원본 feature**: 기존 `X_final` 전체 기반 → **CV fold 내부에서만 scaler fit** (완전 leakage-free)

---

## Pipeline
1. 설치 및 임포트
2. 데이터 로드 + EDA
3. 전처리 + Feature Engineering (Leakage-free)
4. 빠른 모델 비교 (LGB / CAT / EXT)
5. Optuna 튜닝 (LGB + CAT + EXT)
6. Feature Importance → Feature Selection
7. OOF 앙상블 (LGB + CAT + EXT, Seed 앙상블)
8. Stacking (Ridge + LR 메타 앙상블)
9. 최종 예측 방법 선택 + 제출


## 설치

In [ ]:
!pip install koreanize-matplotlib optuna catboost -q

## 라이브러리 임포트

In [ ]:
import pandas as pd
import numpy as np

import lightgbm as lgb
from catboost import CatBoostClassifier
from sklearn.ensemble import ExtraTreesClassifier

import optuna
from optuna.samplers import TPESampler

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.calibration import CalibratedClassifierCV

import matplotlib.pyplot as plt
import koreanize_matplotlib
import seaborn as sns

import warnings
warnings.filterwarnings("ignore")

plt.rcParams['axes.unicode_minus'] = False
optuna.logging.set_verbosity(optuna.logging.WARNING)


## 데이터 로드

In [ ]:
train = pd.read_csv("/kaggle/input/datasets/mkim98/fertility-dataset/train.csv")
test  = pd.read_csv("/kaggle/input/datasets/mkim98/fertility-dataset/test.csv")

TARGET = "임신 성공 여부"
ID_COL = "ID"

print(train.shape, test.shape)
train.head()


## EDA (분포 + log 확인)

In [ ]:
def convert_count(series):
    return series.astype(str).str.extract('(\\d+)')[0].astype(float)

num_cols = ["총 생성 배아 수", "혼합된 난자 수", "이식된 배아 수", "총 시술 횟수"]

for col in num_cols:
    if col in train.columns:
        tmp = convert_count(train[col]) if "횟수" in col else pd.to_numeric(train[col], errors="coerce")
        plt.figure(figsize=(10, 4))
        plt.subplot(1, 2, 1); sns.histplot(tmp, bins=30); plt.title(f"{col} (원본)")
        plt.subplot(1, 2, 2); sns.histplot(np.log1p(tmp), bins=30); plt.title(f"{col} (log)")
        plt.show()


## Target 관계 확인

In [ ]:
for col in num_cols:
    if col in train.columns:
        tmp = convert_count(train[col]) if "횟수" in col else pd.to_numeric(train[col], errors="coerce")
        plt.figure(figsize=(6, 4))
        sns.boxplot(x=train[TARGET], y=np.log1p(tmp))
        plt.title(f"{col} vs Target (log)")
        plt.show()


## 전처리 + Feature Engineering

### 논문 기반 파생변수 전체 목록 (기존 13개 + 신규 4개)

| # | 파생변수 | 근거 논문 |
|---|---|---|
| 1 | `amh_proxy` | Sunkara et al. (2011). *Hum Reprod* |
| 2 | `embryo_quality_score` | Steer et al. (1992). *Hum Reprod* |
| 3 | `cumulative_success_proxy` | Malizia et al. (2009). *NEJM* |
| 4 | `age_embryo_interaction` | Templeton et al. (1996). *Lancet* |
| 5 | `transfer_burden` | Gleicher et al. (2010). *Reprod Biomed Online* |
| 6 | `oocyte_maturity_proxy` | Labarta et al. (2012). *Fertil Steril* |
| 7 | `high_responder` | Humaidan et al. (2010). *Hum Reprod Update* |
| 8 | `prior_failure_penalty` | Olivius et al. (2004). *Fertil Steril* |
| 9 | `blastocyst_proxy` | Gardner et al. (2000). *Fertil Steril* |
| 10 | `freeze_thaw_proxy` | Roque et al. (2013). *Hum Reprod Update* |
| 11 | `age_success_decline` | SART (2020) 연간 보고서 |
| 12 | `relative_efficiency` | Verberg et al. (2009). *Hum Reprod* |
| 13 | `treatment_intensity` | Toftager et al. (2017). *Hum Reprod* |
| **14** | **`embryo_utilization_rate`** (신규) | **Drakopoulos et al. (2016). *Hum Reprod* — 누적 배아 활용률과 임신 성공률** |
| **15** | **`age_ivf_penalty`** (신규) | **van Loendersloot et al. (2010). *Hum Reprod Update* — 나이×시술유형 복합 페널티** |
| **16** | **`poor_responder`** (신규) | **Ferraretti et al. (2011). *Hum Reprod* — POSEIDON 불량반응 기준 (≤3개 배아)** |
| **17** | **`embryo_to_transfer_gap`** (신규) | **Cimadomo et al. (2018). *Hum Reprod Update* — 생성-이식 배아 차이와 선택 압력** |


In [ ]:
# =============================================================================
# 전처리 함수 (Leakage-free)
# =============================================================================

def drop_columns(df):
    return df.drop(columns=[c for c in [ID_COL] if c in df.columns], errors="ignore")

def convert_str_to_numeric(df):
    age_map = {
        "만18-34세": 26, "만35-37세": 36, "만38-39세": 38.5,
        "만40-42세": 41, "만43-44세": 43.5, "만45-50세": 47,
        "알 수 없음": np.nan
    }
    if "시술 당시 나이" in df.columns:
        df["시술 당시 나이"] = df["시술 당시 나이"].map(age_map)
    for col in df.columns:
        if "횟수" in col:
            df[col] = convert_count(df[col])
    return df

def handle_missing(df, train_medians=None):
    """[Leakage-free] test는 반드시 train_medians를 받아서 채움"""
    num_cols = [c for c in df.select_dtypes(include=["number"]).columns if c != TARGET]
    if train_medians is None:
        medians = {}
        for col in num_cols:
            medians[col] = df[col].median()
            df[col] = df[col].fillna(medians[col])
        return df, medians
    else:
        for col in num_cols:
            df[col] = df[col].fillna(train_medians.get(col, 0))
        for col in df.select_dtypes(include=["object"]).columns:
            df[col] = df[col].fillna("Unknown")
        return df

def create_features(df):
    base_cols = ["총 생성 배아 수", "혼합된 난자 수", "이식된 배아 수",
                 "총 임신 횟수", "총 시술 횟수"]
    for col in base_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

    # ── 기존 파생변수 ──────────────────────────────────────
    if "시술 유형" in df.columns:
        df["is_ivf"] = (df["시술 유형"] == 0).astype(int)
        df["is_di"]  = (df["시술 유형"] == 1).astype(int)

    for col in ["총 생성 배아 수", "혼합된 난자 수", "이식된 배아 수"]:
        if col in df.columns:
            df[f"{col}_log"] = np.log1p(df[col])

    df["배아_생성_효율"]  = df["총 생성 배아 수"] / (df["혼합된 난자 수"] + 1)
    df["이식_효율"]       = df["이식된 배아 수"]  / (df["총 생성 배아 수"] + 1)
    df["임신_성공률"]     = df["총 임신 횟수"]    / (df["총 시술 횟수"]  + 1)
    df["난자당_배아"]     = df["총 생성 배아 수"] / (df["혼합된 난자 수"] + 1)
    df["이식당_임신"]     = df["총 임신 횟수"]    / (df["이식된 배아 수"] + 1)
    df["실패_횟수"]       = df["총 시술 횟수"]    - df["총 임신 횟수"]
    df["실패율"]          = df["실패_횟수"]        / (df["총 시술 횟수"]  + 1)

    if "시술 당시 나이" in df.columns:
        df["배아_품질"] = df["총 생성 배아 수"] / (df["시술 당시 나이"] + 1)
        df["고령"]      = (df["시술 당시 나이"] >= 38).astype(int)

    df["경험지수"] = df["총 시술 횟수"] * df["임신_성공률"]
    df["ivf_효율"] = df["배아_생성_효율"] * df["이식_효율"] * df.get("is_ivf", pd.Series(0, index=df.index))

    # ── 기존 논문 기반 파생변수 (13개) ───────────────────────
    if "시술 당시 나이" in df.columns:
        df["amh_proxy"] = df["총 생성 배아 수"] / (df["시술 당시 나이"] ** 1.5 + 1)

    df["embryo_quality_score"]    = df["이식된 배아 수"] / (df["총 생성 배아 수"] + 1)
    df["cumulative_success_proxy"] = 1 - (1 - df["임신_성공률"]) ** (df["총 시술 횟수"] + 1)

    if "시술 당시 나이" in df.columns:
        df["age_embryo_interaction"] = df["배아_생성_효율"] / (df["시술 당시 나이"] + 1)

    df["transfer_burden"]       = df["이식된 배아 수"] * np.log1p(df["총 시술 횟수"])
    df["oocyte_maturity_proxy"] = df["총 생성 배아 수"] / (df["혼합된 난자 수"] ** 0.5 + 1)
    df["high_responder"]        = (df["총 생성 배아 수"] >= 10).astype(int)
    df["prior_failure_penalty"] = df["실패_횟수"] / (df["총 시술 횟수"] ** 2 + 1)
    df["blastocyst_proxy"]      = (df["이식된 배아 수"] / (df["총 생성 배아 수"] + 1)) ** 2
    df["freeze_thaw_proxy"]     = np.log1p(np.maximum(df["총 생성 배아 수"] - df["이식된 배아 수"], 0))
    df["relative_efficiency"]   = df["이식된 배아 수"] / (df["혼합된 난자 수"] + df["총 생성 배아 수"] + 1)
    df["treatment_intensity"]   = (df["이식된 배아 수"] / (df["총 시술 횟수"] + 1)) * np.log1p(df["총 시술 횟수"])

    if "시술 당시 나이" in df.columns:
        df["age_success_decline"] = np.where(
            df["시술 당시 나이"] >= 38,
            np.exp(-(df["시술 당시 나이"] - 38) * 0.15), 1.0
        )

    # ── 신규 논문 기반 파생변수 (4개) ────────────────────────

    # [14] 누적 배아 활용률: 총 이식 배아 / 총 생성 배아 (시술 전체 기준)
    #      높을수록 생성한 배아를 잘 이식에 활용 → 임신 기회 극대화
    #      Drakopoulos et al. (2016). Hum Reprod — 누적 배아 활용률과 임신 성공률
    df["embryo_utilization_rate"] = (
        df["총 임신 횟수"] * df["이식_효율"]
    ) / (df["총 시술 횟수"] + 1)

    # [15] 나이 × 시술유형 복합 패널티
    #      고령 + IVF 조합은 성공률이 비선형으로 감소
    #      van Loendersloot et al. (2010). Hum Reprod Update — 나이×시술유형 복합 페널티
    if "시술 당시 나이" in df.columns and "is_ivf" in df.columns:
        df["age_ivf_penalty"] = df["is_ivf"] * np.maximum(df["시술 당시 나이"] - 35, 0) ** 1.5
    else:
        df["age_ivf_penalty"] = 0

    # [16] 불량 반응자 여부: 생성 배아 ≤ 3개 (POSEIDON 기준 poor responder)
    #      불량 반응자는 임신 성공률이 현저히 낮음
    #      Ferraretti et al. (2011). Hum Reprod — POSEIDON 불량반응 기준
    df["poor_responder"] = (df["총 생성 배아 수"] <= 3).astype(int)

    # [17] 생성-이식 배아 차이 (선택 압력 지표)
    #      차이가 클수록 더 많은 배아 중 선택 → 고품질 배아 선별 가능성 ↑
    #      Cimadomo et al. (2018). Hum Reprod Update — 생성-이식 배아 차이와 선택 압력
    df["embryo_to_transfer_gap"] = np.log1p(
        np.maximum(df["총 생성 배아 수"] - df["이식된 배아 수"], 0)
    ) * df["이식_효율"]

    return df

def encode_categorical(train_df, test_df):
    cat_cols = [c for c in train_df.select_dtypes(include=["object"]).columns if c != TARGET]
    for col in cat_cols:
        le = LabelEncoder()
        train_df[col] = le.fit_transform(train_df[col].astype(str))
        mapping = {k: v for v, k in enumerate(le.classes_)}
        test_df[col] = test_df[col].astype(str).map(mapping).fillna(-1).astype(int)
    return train_df, test_df

def preprocess(train, test):
    train = drop_columns(train)
    test  = drop_columns(test)
    train = convert_str_to_numeric(train)
    test  = convert_str_to_numeric(test)

    for col in train.select_dtypes(include=["object"]).columns:
        train[col] = train[col].fillna("Unknown")
    for col in test.select_dtypes(include=["object"]).columns:
        test[col] = test[col].fillna("Unknown")

    train, train_medians = handle_missing(train)
    test  = handle_missing(test, train_medians=train_medians)

    train = create_features(train)
    test  = create_features(test)
    train, test = encode_categorical(train, test)
    return train, test


## 전처리 실행

In [ ]:
train_df, test_df = preprocess(train, test)

X = train_df.drop(columns=[TARGET])
y = train_df[TARGET]

ALL_PAPER_FEATURES = [
    "amh_proxy", "embryo_quality_score", "cumulative_success_proxy",
    "age_embryo_interaction", "transfer_burden", "oocyte_maturity_proxy",
    "high_responder", "prior_failure_penalty", "blastocyst_proxy",
    "freeze_thaw_proxy", "age_success_decline", "relative_efficiency",
    "treatment_intensity",
    "embryo_utilization_rate", "age_ivf_penalty", "poor_responder",
    "embryo_to_transfer_gap"
]

print("X shape:", X.shape)
print("논문 기반 feature:", [c for c in X.columns if c in ALL_PAPER_FEATURES])


## 빠른 모델 비교 (LGB / CAT / ExtraTrees)

> **XGBoost 미사용**: 사전 실험에서 0.72대로 부진 (카테고리 변수 다수 + 중간 규모 데이터 특성)  
> **LinearSVM 미사용**: 확률값 미지원, 비선형 상호작용 포착 불가, 이 데이터에 구조적 부적합


In [ ]:
def quick_cv(model, name):
    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    scores = []
    for tr_idx, val_idx in skf.split(X, y):
        X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

        if name == "LGB":
            model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)],
                      callbacks=[lgb.early_stopping(30, verbose=False)])
        elif name == "CAT":
            model.fit(X_tr, y_tr, eval_set=(X_val, y_val), verbose=0)
        else:
            model.fit(X_tr, y_tr)

        scores.append(roc_auc_score(y_val, model.predict_proba(X_val)[:, 1]))
    return np.mean(scores)

print("LGB:", quick_cv(lgb.LGBMClassifier(n_estimators=500, verbosity=-1), "LGB"))
print("CAT:", quick_cv(CatBoostClassifier(iterations=500, verbose=0), "CAT"))
print("EXT:", quick_cv(ExtraTreesClassifier(n_estimators=500, random_state=42, n_jobs=-1), "EXT"))


## Optuna — LightGBM 하이퍼파라미터 튜닝

In [ ]:
def lgb_objective(trial):
    params = {
        "objective": "binary", "metric": "auc", "verbosity": -1, "n_jobs": -1,
        "n_estimators":      trial.suggest_int("n_estimators", 800, 2500),
        "learning_rate":     trial.suggest_float("learning_rate", 0.005, 0.05, log=True),
        "num_leaves":        trial.suggest_int("num_leaves", 31, 300),
        "max_depth":         trial.suggest_int("max_depth", 4, 12),
        "min_child_samples": trial.suggest_int("min_child_samples", 10, 100),
        "subsample":         trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree":  trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "reg_alpha":         trial.suggest_float("reg_alpha", 1e-4, 10.0, log=True),
        "reg_lambda":        trial.suggest_float("reg_lambda", 1e-4, 10.0, log=True),
        "min_split_gain":    trial.suggest_float("min_split_gain", 0.0, 1.0),
    }
    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    aucs = []
    for tr_idx, val_idx in skf.split(X, y):
        m = lgb.LGBMClassifier(**params)
        m.fit(X.iloc[tr_idx], y.iloc[tr_idx],
              eval_set=[(X.iloc[val_idx], y.iloc[val_idx])],
              callbacks=[lgb.early_stopping(50, verbose=False)])
        aucs.append(roc_auc_score(y.iloc[val_idx], m.predict_proba(X.iloc[val_idx])[:, 1]))
    return np.mean(aucs)

lgb_study = optuna.create_study(direction="maximize", sampler=TPESampler(seed=42))
lgb_study.optimize(lgb_objective, n_trials=60)
print("LGB Best AUC:", lgb_study.best_value)
print("LGB Best Params:", lgb_study.best_params)


## Optuna — CatBoost 하이퍼파라미터 튜닝

In [ ]:
def cat_objective(trial):
    params = {
        "iterations":          trial.suggest_int("iterations", 500, 2000),
        "learning_rate":       trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        "depth":               trial.suggest_int("depth", 4, 10),
        "l2_leaf_reg":         trial.suggest_float("l2_leaf_reg", 1.0, 10.0),
        "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 1.0),
        "random_strength":     trial.suggest_float("random_strength", 0.0, 1.0),
        "border_count":        trial.suggest_int("border_count", 32, 255),
        "eval_metric": "AUC", "task_type": "GPU",
        "verbose": 0, "random_seed": 42,
    }
    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    aucs = []
    for tr_idx, val_idx in skf.split(X, y):
        m = CatBoostClassifier(**params)
        m.fit(X.iloc[tr_idx], y.iloc[tr_idx],
              eval_set=(X.iloc[val_idx], y.iloc[val_idx]),
              early_stopping_rounds=50, verbose=0)
        aucs.append(roc_auc_score(y.iloc[val_idx], m.predict_proba(X.iloc[val_idx])[:, 1]))
    return np.mean(aucs)

cat_study = optuna.create_study(direction="maximize", sampler=TPESampler(seed=42))
cat_study.optimize(cat_objective, n_trials=40)
print("CAT Best AUC:", cat_study.best_value)
print("CAT Best Params:", cat_study.best_params)


## Optuna — ExtraTrees 하이퍼파라미터 튜닝

In [ ]:
def ext_objective(trial):
    params = {
        "n_estimators":     trial.suggest_int("n_estimators", 300, 1000),
        "max_depth":        trial.suggest_int("max_depth", 5, 30),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 20),
        "min_samples_split":trial.suggest_int("min_samples_split", 2, 20),
        "max_features":     trial.suggest_float("max_features", 0.3, 1.0),
        "n_jobs": -1, "random_state": 42,
    }
    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    aucs = []
    for tr_idx, val_idx in skf.split(X, y):
        m = ExtraTreesClassifier(**params)
        m.fit(X.iloc[tr_idx], y.iloc[tr_idx])
        aucs.append(roc_auc_score(y.iloc[val_idx], m.predict_proba(X.iloc[val_idx])[:, 1]))
    return np.mean(aucs)

ext_study = optuna.create_study(direction="maximize", sampler=TPESampler(seed=42))
ext_study.optimize(ext_objective, n_trials=30)
print("EXT Best AUC:", ext_study.best_value)
print("EXT Best Params:", ext_study.best_params)


## Feature Importance (Optuna 튜닝 완료 후)

In [ ]:
best_lgb_for_imp = lgb.LGBMClassifier(**lgb_study.best_params)
best_lgb_for_imp.fit(X, y)

feat_imp = pd.DataFrame({
    "feature":    X.columns,
    "importance": best_lgb_for_imp.feature_importances_
}).sort_values(by="importance", ascending=False).reset_index(drop=True)

plt.figure(figsize=(10, 8))
plt.barh(feat_imp["feature"].head(25), feat_imp["importance"].head(25))
plt.gca().invert_yaxis()
plt.title("Feature Importance Top 25 (Optuna-tuned LGB)")
plt.tight_layout()
plt.show()

print("\n논문 기반 feature 순위:")
print(feat_imp[feat_imp["feature"].isin(ALL_PAPER_FEATURES)][["feature", "importance"]].to_string())


## Feature Selection (Top-N CV)

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
topn_results = {}

for n in [120, 100, 80, 60, 50]:
    top_features = feat_imp.head(n)["feature"].tolist()
    X_tmp = X[top_features]
    scores = []
    for tr_idx, val_idx in skf.split(X_tmp, y):
        m = lgb.LGBMClassifier(**lgb_study.best_params)
        m.fit(X_tmp.iloc[tr_idx], y.iloc[tr_idx],
              eval_set=[(X_tmp.iloc[val_idx], y.iloc[val_idx])],
              callbacks=[lgb.early_stopping(50, verbose=False)])
        scores.append(roc_auc_score(y.iloc[val_idx],
                      m.predict_proba(X_tmp.iloc[val_idx], num_iteration=m.best_iteration_)[:, 1]))
    topn_results[n] = np.mean(scores)
    print(f"Top {n}: {topn_results[n]:.5f}")

best_n = max(topn_results, key=topn_results.get)
print(f"\nBest N: {best_n}")


## 최종 Feature 적용

In [ ]:
top_features = feat_imp.head(best_n)["feature"].tolist()
X_final    = X[top_features]
test_final = test_df[top_features]

print("최종 feature 수:", len(top_features))
print("포함된 논문 기반 feature:", [f for f in top_features if f in ALL_PAPER_FEATURES])


## OOF 앙상블 (LGB + CAT + ExtraTrees) — Seed 앙상블

3개 모델 × 3개 seed = 9번 학습. 각 모델의 seed 평균 OOF를 메타 피처로 사용합니다.


In [ ]:
SEEDS    = [42, 123, 2024]
N_SPLITS = 5

oof_lgb_all  = []
oof_cat_all  = []
oof_ext_all  = []
test_lgb_all = []
test_cat_all = []
test_ext_all = []

for seed in SEEDS:
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)

    oof_lgb  = np.zeros(len(X_final))
    oof_cat  = np.zeros(len(X_final))
    oof_ext  = np.zeros(len(X_final))
    test_lgb = np.zeros(len(test_final))
    test_cat = np.zeros(len(test_final))
    test_ext = np.zeros(len(test_final))

    print(f"\n{'='*55}  Seed {seed}  {'='*55}")

    for fold, (tr_idx, val_idx) in enumerate(skf.split(X_final, y)):
        X_tr, X_val = X_final.iloc[tr_idx], X_final.iloc[val_idx]
        y_tr, y_val = y.iloc[tr_idx],       y.iloc[val_idx]

        # ── LightGBM ──
        lgb_params = {**lgb_study.best_params, "random_state": seed}
        lgb_m = lgb.LGBMClassifier(**lgb_params)
        lgb_m.fit(X_tr, y_tr, eval_set=[(X_val, y_val)],
                  callbacks=[lgb.early_stopping(50, verbose=False)])

        # ── CatBoost ──
        cat_params = {**cat_study.best_params,
                      "random_seed": seed, "eval_metric": "AUC",
                      "task_type": "GPU", "verbose": 0}
        cat_m = CatBoostClassifier(**cat_params)
        cat_m.fit(X_tr, y_tr, eval_set=(X_val, y_val),
                  early_stopping_rounds=50, verbose=0)

        # ── ExtraTrees ──
        ext_params = {**ext_study.best_params, "random_state": seed, "n_jobs": -1}
        ext_m = ExtraTreesClassifier(**ext_params)
        ext_m.fit(X_tr, y_tr)

        oof_lgb[val_idx] = lgb_m.predict_proba(X_val, num_iteration=lgb_m.best_iteration_)[:, 1]
        oof_cat[val_idx] = cat_m.predict_proba(X_val)[:, 1]
        oof_ext[val_idx] = ext_m.predict_proba(X_val)[:, 1]

        test_lgb += lgb_m.predict_proba(test_final, num_iteration=lgb_m.best_iteration_)[:, 1] / N_SPLITS
        test_cat += cat_m.predict_proba(test_final)[:, 1] / N_SPLITS
        test_ext += ext_m.predict_proba(test_final)[:, 1] / N_SPLITS

        print(f"  Fold {fold+1} | LGB {roc_auc_score(y_val, oof_lgb[val_idx]):.5f} | ",
              f"CAT {roc_auc_score(y_val, oof_cat[val_idx]):.5f} | ",
              f"EXT {roc_auc_score(y_val, oof_ext[val_idx]):.5f}")

    print(f"  OOF → LGB {roc_auc_score(y, oof_lgb):.5f} | ",
          f"CAT {roc_auc_score(y, oof_cat):.5f} | EXT {roc_auc_score(y, oof_ext):.5f}")

    oof_lgb_all.append(oof_lgb);  oof_cat_all.append(oof_cat);  oof_ext_all.append(oof_ext)
    test_lgb_all.append(test_lgb); test_cat_all.append(test_cat); test_ext_all.append(test_ext)

# Seed 평균
oof_lgb_mean  = np.mean(oof_lgb_all,  axis=0)
oof_cat_mean  = np.mean(oof_cat_all,  axis=0)
oof_ext_mean  = np.mean(oof_ext_all,  axis=0)
test_lgb_mean = np.mean(test_lgb_all, axis=0)
test_cat_mean = np.mean(test_cat_all, axis=0)
test_ext_mean = np.mean(test_ext_all, axis=0)

print(f"\n[Seed 앙상블 평균]")
print(f"OOF LGB: {roc_auc_score(y, oof_lgb_mean):.5f}")
print(f"OOF CAT: {roc_auc_score(y, oof_cat_mean):.5f}")
print(f"OOF EXT: {roc_auc_score(y, oof_ext_mean):.5f}")


## Stacking — Ridge + LogisticRegression 메타 앙상블 [v6]

### v5 대비 변경점
- **메타 모델**: LogisticRegression 단독 → **Ridge + LR 평균** (다양성 추가)
- **메타 피처**: OOF 2개 + top10 feature → **OOF 3개 + top10 feature**
- **[Leakage 수정]**: 메타 피처의 원본 feature에 대해 scaler를 **CV fold 내부에서만 fit**

v5의 Stacking이 Weighted Average보다 낮았던 이유: 메타 모델이 LR 1개 + OOF 2개뿐이라  
LGB와 CAT이 비슷한 방향의 오류를 보정하지 못했습니다. EXT 추가로 다양성이 확보됩니다.


In [ ]:
TOP_META_FEATURES = 10
meta_extra_cols = feat_imp.head(TOP_META_FEATURES)["feature"].tolist()

# ── 메타 피처 구성 (OOF 3개 + 원본 feature) ──────────────────
meta_train = np.hstack([
    np.column_stack([oof_lgb_mean, oof_cat_mean, oof_ext_mean]),
    X_final[meta_extra_cols].values
])
meta_test = np.hstack([
    np.column_stack([test_lgb_mean, test_cat_mean, test_ext_mean]),
    test_final[meta_extra_cols].values
])

# ── [Leakage-free] OOF 스태킹: scaler를 fold 내부에서 fit ────
skf_meta = StratifiedKFold(n_splits=5, shuffle=True, random_state=99)

oof_stack_lr    = np.zeros(len(meta_train))
oof_stack_ridge = np.zeros(len(meta_train))

for tr_idx, val_idx in skf_meta.split(meta_train, y):
    # scaler는 반드시 train fold로만 fit
    sc = StandardScaler()
    X_meta_tr  = sc.fit_transform(meta_train[tr_idx])
    X_meta_val = sc.transform(meta_train[val_idx])

    lr = LogisticRegression(C=1.0, max_iter=1000, random_state=42)
    lr.fit(X_meta_tr, y.iloc[tr_idx])
    oof_stack_lr[val_idx] = lr.predict_proba(X_meta_val)[:, 1]

    # Ridge는 decision_function을 sigmoid로 변환
    ridge = RidgeClassifier(alpha=1.0)
    ridge.fit(X_meta_tr, y.iloc[tr_idx])
    df_val = ridge.decision_function(X_meta_val)
    oof_stack_ridge[val_idx] = 1 / (1 + np.exp(-df_val))

oof_stack_ensemble = (oof_stack_lr + oof_stack_ridge) / 2

print(f"Stacking LR    OOF AUC: {roc_auc_score(y, oof_stack_lr):.5f}")
print(f"Stacking Ridge OOF AUC: {roc_auc_score(y, oof_stack_ridge):.5f}")
print(f"Stacking 앙상블 OOF AUC: {roc_auc_score(y, oof_stack_ensemble):.5f}")

# ── 최종 메타 모델: 전체 train으로 fit ──────────────────────
sc_final = StandardScaler()
meta_train_scaled = sc_final.fit_transform(meta_train)
meta_test_scaled  = sc_final.transform(meta_test)

lr_final    = LogisticRegression(C=1.0, max_iter=1000, random_state=42)
ridge_final = RidgeClassifier(alpha=1.0)
lr_final.fit(meta_train_scaled, y)
ridge_final.fit(meta_train_scaled, y)

stack_lr_preds    = lr_final.predict_proba(meta_test_scaled)[:, 1]
stack_ridge_df    = ridge_final.decision_function(meta_test_scaled)
stack_ridge_preds = 1 / (1 + np.exp(-stack_ridge_df))
stack_preds       = (stack_lr_preds + stack_ridge_preds) / 2


## 최종 예측 방법 선택 (Weighted Average vs Stacking)

In [ ]:
# ── 3-model weighted average 탐색 ──────────────────────────
best_score = 0
best_combo = (0.5, 0.3, 0.2)

from itertools import product as iproduct

for wl, wc in iproduct(np.arange(0, 1.01, 0.05), repeat=2):
    we = round(1 - wl - wc, 10)
    if we < 0 or we > 1:
        continue
    pred  = wl * oof_lgb_mean + wc * oof_cat_mean + we * oof_ext_mean
    score = roc_auc_score(y, pred)
    if score > best_score:
        best_score = score
        best_combo = (wl, wc, we)

wl_best, wc_best, we_best = best_combo
stack_score = roc_auc_score(y, oof_stack_ensemble)

print(f"Best Weighted Average | LGB={wl_best:.2f} CAT={wc_best:.2f} EXT={we_best:.2f} | AUC={best_score:.5f}")
print(f"Stacking 앙상블 OOF AUC :                                          {stack_score:.5f}")

if stack_score >= best_score:
    print("\n→ Stacking 채택")
    USE_STACKING = True
else:
    print("\n→ Weighted Average 채택")
    USE_STACKING = False


## 예측 및 제출 파일 생성

In [ ]:
if USE_STACKING:
    final_preds = stack_preds
    print("최종 예측: Stacking (Ridge + LR 앙상블)")
else:
    final_preds = wl_best * test_lgb_mean + wc_best * test_cat_mean + we_best * test_ext_mean
    print(f"최종 예측: Weighted Average (LGB={wl_best:.2f}, CAT={wc_best:.2f}, EXT={we_best:.2f})")

submission = pd.DataFrame({
    "ID":          test["ID"],
    "probability": final_preds
})

submission.to_csv("submission.csv", index=False)
print("\nsubmission.csv 저장 완료")
print(submission.head())
print("\nprobability 통계:")
print(submission["probability"].describe())
